# Accretion Disk / Relativistic Jet Launch

In [7]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFilter
from vizlib.animation_export import export_animation


# ============================================================
# Relativistic Jet Launch
# Accretion disk + Kerr-like jet system
# ============================================================

OUTPUT_FORMAT = "webm"  # gif | mp4 | webm
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "relativistic_jet_launch"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

CX = WIDTH * 0.50
CY = HEIGHT * 0.52

BH_R = 34

DISK_RX = 260
DISK_RY = 74

JET_LENGTH = 420
JET_WIDTH = 46

STAR_COUNT = 380
DISK_PARTICLES = 2400
JET_PARTICLES = 1800

FRAME_DRAG_STRENGTH = 0.45


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.35, 1.45),
        rng.uniform(18, 90),
        rng.uniform(0.35, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:

        tw = (
            0.55
            + 0.45 * np.sin(
                phase * 2 * np.pi * speed + ph
            ) ** 2
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# SPACE-TIME SWIRL
# ============================================================

def draw_frame_dragging(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for k in range(28):

        theta = np.linspace(
            0,
            2 * np.pi,
            1200,
        )

        rr = (
            55
            + k * 11
        )

        swirl = (
            FRAME_DRAG_STRENGTH
            * np.exp(-rr / 180)
        )

        angle = (
            theta
            + swirl * theta
            + phase * 2 * np.pi * 0.35
        )

        x = CX + rr * np.cos(angle)
        y = CY + rr * 0.42 * np.sin(angle)

        alpha = int(
            4
            + 18 * np.exp(-k / 8)
        )

        d.line(
            list(zip(x, y)),
            fill=(90, 220, 255, alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=5)


# ============================================================
# ACCRETION DISK
# ============================================================

disk_particles = []

for _ in range(DISK_PARTICLES):

    disk_particles.append((
        rng.uniform(0, 2 * np.pi),
        rng.uniform(0.0, 1.0),
        rng.uniform(0.4, 1.5),
        rng.uniform(0.5, 2.0),
        rng.uniform(0.3, 1.0),
    ))


def draw_disk(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    # volumetric disk body
    for scale, alpha in [
        (1.00, 14),
        (0.82, 28),
        (0.64, 42),
    ]:

        d.ellipse(
            [
                CX - DISK_RX * scale,
                CY - DISK_RY * scale,
                CX + DISK_RX * scale,
                CY + DISK_RY * scale,
            ],
            outline=(120, 220, 255, alpha),
            width=int(20 * scale),
        )

    # bright inner ring
    d.ellipse(
        [
            CX - 88,
            CY - 28,
            CX + 88,
            CY + 28,
        ],
        outline=(255, 220, 120, 180),
        width=5,
    )

    # particles
    for angle0, radial, speed, size, brightness in disk_particles:

        rr = (
            75
            + radial * (DISK_RX - 80)
        )

        local_speed = (
            2.8 / np.sqrt(rr / 60)
        )

        angle = (
            angle0
            + phase * speed * local_speed * 2 * np.pi
        )

        # relativistic asymmetry
        doppler = (
            0.5
            + 0.5 * np.cos(angle)
        )

        x = CX + rr * np.cos(angle)

        y = (
            CY
            + rr * 0.28 * np.sin(angle)
        )

        # frame dragging distortion
        x += (
            12
            * np.exp(-rr / 120)
            * np.sin(angle * 2 + phase * 6)
        )

        alpha = int(
            40
            + 160 * brightness * doppler
        )

        col = (
            255,
            int(140 + 80 * doppler),
            int(60 + 40 * doppler),
            alpha,
        )

        rr2 = size

        d.ellipse(
            [
                x - rr2,
                y - rr2,
                x + rr2,
                y + rr2,
            ],
            fill=col,
        )

    add_glow(base, layer, blur=8)


# ============================================================
# RELATIVISTIC JETS
# ============================================================

jet_particles = []

for _ in range(JET_PARTICLES):

    jet_particles.append((
        rng.choice([-1, 1]),
        rng.uniform(0.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(0.6, 2.0),
        rng.uniform(0.5, 2.2),
    ))


def draw_jets(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    # volumetric cones
    for sign in [-1, 1]:

        theta = np.linspace(
            -1.0,
            1.0,
            260,
        )

        x = (
            CX
            + theta * (
                JET_WIDTH
                + np.abs(theta) * 18
            )
        )

        y = (
            CY
            + sign * (
                20
                + np.abs(theta) * JET_LENGTH
            )
        )

        pts = list(zip(x, y))

        for width, alpha in [
            (54, 6),
            (32, 16),
            (18, 32),
            (7, 90),
        ]:

            d.line(
                pts,
                fill=(120, 240, 255, alpha),
                width=width,
                joint="curve",
            )

    # particles
    for sign, offset, spread, speed, size in jet_particles:

        t = (
            offset
            + phase * speed
        ) % 1.0

        y = (
            CY
            + sign * (
                30
                + t * JET_LENGTH
            )
        )

        width = (
            8
            + t * JET_WIDTH
        )

        x = (
            CX
            + spread * width
            + 8 * np.sin(
                t * 8
                + phase * 2 * np.pi * 3
            )
        )

        alpha = int(
            60
            + 140 * (1 - abs(spread))
        )

        rr = size * (
            1.0 + 0.4 * t
        )

        d.ellipse(
            [
                x - rr,
                y - rr,
                x + rr,
                y + rr,
            ],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=10)


# ============================================================
# MAGNETIC FIELD STRUCTURE
# ============================================================

def draw_magnetic_fields(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    for sign in [-1, 1]:

        for k in range(12):

            t = np.linspace(
                0,
                1,
                220,
            )

            radius = (
                18
                + k * 8
            )

            x = (
                CX
                + radius * np.cos(
                    t * 6
                    + phase * 2 * np.pi * 0.7
                )
            )

            y = (
                CY
                + sign * t * JET_LENGTH * 0.85
            )

            pts = list(zip(x, y))

            d.line(
                pts,
                fill=(90, 220, 255, 18),
                width=1,
                joint="curve",
            )

    add_glow(base, layer, blur=4)


# ============================================================
# BLACK HOLE
# ============================================================

def draw_black_hole(base):

    layer = rgba()
    d = ImageDraw.Draw(layer)


    # shadow
    d.ellipse(
        [
            CX - BH_R,
            CY - BH_R,
            CX + BH_R,
            CY + BH_R,
        ],
        fill=(0, 0, 0, 255),
    )

    add_glow(base, layer, blur=5)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new(
        "RGBA",
        (WIDTH, HEIGHT),
        BG,
    )

    draw_background(base, phase)

    draw_frame_dragging(base, phase)

    draw_magnetic_fields(base, phase)

    draw_jets(base, phase)

    draw_disk(base, phase)

    draw_black_hole(base)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Relativistic Jet Launch")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(
            f"[RENDER] frame {i}/{TOTAL_FRAMES}"
        )

    frames.append(render_frame(i))


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
)

print()
print("[OUT_FILE]", out_file)
print("[EXISTS]", out_file.exists())
print("[SIZE]", out_file.stat().st_size if out_file.exists() else "missing")

[START] Relativistic Jet Launch
[RENDER] frame 0/192
[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[OUT_FILE] animations/relativistic_jet_launch/relativistic_jet_launch.webm
[EXISTS] True
[SIZE] 3162801


[out#0/webm @ 0x13461bcc0] video:3079KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.329684%
frame=  192 fps= 25 q=32.0 Lsize=    3089KiB time=00:00:08.00 bitrate=3162.8kbits/s speed=1.05x    


# Pulsar Magnetosphere / Rotating Neutron Star

In [8]:
from __future__ import annotations

from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
import imageio.v2 as imageio

from vizlib.animation_export import export_animation


# ============================================================
# Pulsar Magnetosphere
# Rotating neutron star + dipole field
# ============================================================

OUTPUT_FORMAT = "webm"  # gif | mp4 | webm
FPS = 24
DURATION = 8
TOTAL_FRAMES = FPS * DURATION

WIDTH = 960
HEIGHT = 540

ANIMATION_NAME = "pulsar_magnetosphere"

OUT_DIR = Path("media-site/animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

BG = (0, 0, 0, 255)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

CX = WIDTH * 0.50
CY = HEIGHT * 0.52

STAR_R = 34

MAGNETIC_TILT = np.deg2rad(28)
ROTATION_SPEED = 3.6

FIELD_LINE_COUNT = 26
FIELD_PARTICLES = 2400

LIGHT_CYLINDER_R = 220

STAR_COUNT = 340


# ============================================================
# HELPERS
# ============================================================

def rgba():
    return Image.new("RGBA", (WIDTH, HEIGHT), (0, 0, 0, 0))


def add_glow(base, layer, blur=8):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


# ============================================================
# BACKGROUND
# ============================================================

stars = []

for _ in range(STAR_COUNT):

    stars.append((
        rng.uniform(0, WIDTH),
        rng.uniform(0, HEIGHT),
        rng.uniform(0.35, 1.5),
        rng.uniform(15, 85),
        rng.uniform(0.4, 1.1),
        rng.uniform(0, 2 * np.pi),
    ))


def draw_background(base, phase):

    d = ImageDraw.Draw(base)

    for x, y, r, a, speed, ph in stars:

        tw = (
            0.55
            + 0.45 * np.sin(
                phase * 2 * np.pi * speed + ph
            ) ** 2
        )

        d.ellipse(
            [x-r, y-r, x+r, y+r],
            fill=(180, 220, 255, int(a * tw)),
        )


# ============================================================
# LIGHT CYLINDER
# ============================================================

def draw_light_cylinder(base):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    d.ellipse(
        [
            CX - LIGHT_CYLINDER_R,
            CY - LIGHT_CYLINDER_R * 0.32,
            CX + LIGHT_CYLINDER_R,
            CY + LIGHT_CYLINDER_R * 0.32,
        ],
        outline=(100, 220, 255, 28),
        width=2,
    )

    add_glow(base, layer, blur=4)


# ============================================================
# MAGNETIC FIELD LINES
# ============================================================

field_offsets = np.linspace(
    0.18,
    1.25,
    FIELD_LINE_COUNT,
)


def dipole_curve(theta, scale):

    r = scale * np.sin(theta) ** 2

    x = r * np.sin(theta)
    y = r * np.cos(theta)

    return x, y


def rotate_2d(x, y, angle):

    xr = (
        x * np.cos(angle)
        - y * np.sin(angle)
    )

    yr = (
        x * np.sin(angle)
        + y * np.cos(angle)
    )

    return xr, yr


def draw_field_lines(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    spin = (
        phase
        * 2
        * np.pi
        * ROTATION_SPEED
    )

    for scale in field_offsets:

        theta = np.linspace(
            0.02,
            np.pi - 0.02,
            900,
        )

        rscale = (
            60
            + scale * 240
        )

        x, y = dipole_curve(
            theta,
            rscale,
        )

        # flatten perspective
        y *= 0.52

        # rotate magnetic axis
        x, y = rotate_2d(
            x,
            y,
            MAGNETIC_TILT,
        )

        # star rotation
        x, y = rotate_2d(
            x,
            y,
            spin,
        )

        x += CX
        y += CY

        pts = list(zip(x, y))

        alpha = int(
            12
            + 55 * np.exp(-scale * 0.4)
        )

        d.line(
            pts,
            fill=(100, 220, 255, alpha),
            width=1,
            joint="curve",
        )

    add_glow(base, layer, blur=5)


# ============================================================
# SYNCHROTRON PARTICLES
# ============================================================

particles = []

for _ in range(FIELD_PARTICLES):

    particles.append((
        rng.choice(field_offsets),
        rng.uniform(0.05, 0.95),
        rng.uniform(0.4, 1.8),
        rng.uniform(0.6, 2.0),
    ))


def particle_position(scale, t, spin):

    theta = (
        0.05
        + t * (np.pi - 0.1)
    )

    rscale = (
        60
        + scale * 240
    )

    x, y = dipole_curve(
        theta,
        rscale,
    )

    y *= 0.52

    x, y = rotate_2d(
        x,
        y,
        MAGNETIC_TILT,
    )

    x, y = rotate_2d(
        x,
        y,
        spin,
    )

    return x + CX, y + CY


def draw_particles(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    spin = (
        phase
        * 2
        * np.pi
        * ROTATION_SPEED
    )

    for scale, offset, speed, size in particles:

        t = (
            offset
            + phase * speed
        ) % 1.0

        x, y = particle_position(
            scale,
            t,
            spin,
        )

        alpha = int(
            40
            + 150 * (
                1
                - abs(t - 0.5)
            )
        )

        rr = size

        d.ellipse(
            [
                x - rr,
                y - rr,
                x + rr,
                y + rr,
            ],
            fill=(180, 255, 255, alpha),
        )

    add_glow(base, layer, blur=5)


# ============================================================
# RADIO BEAMS
# ============================================================

def draw_beams(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    spin = (
        phase
        * 2
        * np.pi
        * ROTATION_SPEED
    )

    beam_angle = (
        spin + MAGNETIC_TILT
    )

    for direction in [0, np.pi]:

        angle = beam_angle + direction

        dx = xx - CX
        dy = yy - CY

        forward = (
            dx * np.cos(angle)
            + dy * np.sin(angle)
        )

        side = (
            -dx * np.sin(angle)
            + dy * np.cos(angle)
        )

        width = (
            16
            + forward * 0.12
        )

        mask = (
            (forward > 0)
            & (forward < 520)
        )

        radial = np.exp(
            -(side ** 2)
            / (2 * width ** 2)
        )

        fade = (
            np.clip(
                1 - forward / 520,
                0,
                1,
            ) ** 1.8
        )

        alpha = radial * fade * mask

        arr = np.zeros(
            (HEIGHT, WIDTH, 4),
            dtype=np.uint8,
        )

        arr[..., 0] = 110
        arr[..., 1] = 230
        arr[..., 2] = 255

        arr[..., 3] = np.clip(
            alpha * 95,
            0,
            255,
        ).astype(np.uint8)

        beam_img = (
            Image.fromarray(arr, "RGBA")
            .filter(ImageFilter.GaussianBlur(6))
        )

        layer.alpha_composite(beam_img)

    add_glow(base, layer, blur=10)


# ============================================================
# POLAR CAPS
# ============================================================

def draw_polar_caps(base, phase):

    layer = rgba()
    d = ImageDraw.Draw(layer)

    spin = (
        phase
        * 2
        * np.pi
        * ROTATION_SPEED
    )

    for direction in [1, -1]:

        px = (
            direction
            * STAR_R
            * 0.75
        )

        py = 0

        px, py = rotate_2d(
            px,
            py,
            MAGNETIC_TILT,
        )

        px, py = rotate_2d(
            px,
            py,
            spin,
        )

        x = CX + px
        y = CY + py * 0.55

        pulse = (
            0.7
            + 0.3 * np.sin(
                phase * 2 * np.pi * 7
            ) ** 2
        )

        rr = 8 + 4 * pulse

        d.ellipse(
            [
                x - rr,
                y - rr,
                x + rr,
                y + rr,
            ],
            fill=(180, 255, 255, 170),
        )

    add_glow(base, layer, blur=8)


# ============================================================
# NEUTRON STAR
# ============================================================

def draw_star(base, phase):

    layer = rgba()

    yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

    dx = xx - CX
    dy = yy - CY

    r = np.sqrt(dx * dx + dy * dy)

    sphere = r <= STAR_R

    shade = np.clip(
        1.0 - r / STAR_R,
        0,
        1,
    )

    texture = (
        0.5
        + 0.2 * np.sin(dx * 0.18 + phase * 7)
        + 0.18 * np.sin(dy * 0.15 - phase * 9)
    )

    texture = np.clip(texture, 0, 1)

    intensity = (
        shade ** 0.42
        * texture
    )

    arr = np.zeros(
        (HEIGHT, WIDTH, 4),
        dtype=np.uint8,
    )

    arr[..., 0] = np.where(
        sphere,
        120 + 80 * intensity,
        0,
    )

    arr[..., 1] = np.where(
        sphere,
        190 + 50 * intensity,
        0,
    )

    arr[..., 2] = np.where(
        sphere,
        255,
        0,
    )

    arr[..., 3] = np.where(
        sphere,
        255,
        0,
    )

    layer.alpha_composite(
        Image.fromarray(
            arr.astype(np.uint8),
            "RGBA",
        )
    )

    d = ImageDraw.Draw(layer)

    for rr, alpha in [
        (88, 10),
        (62, 24),
        (44, 55),
    ]:

        d.ellipse(
            [
                CX - rr,
                CY - rr,
                CX + rr,
                CY + rr,
            ],
            fill=(120, 220, 255, alpha),
        )

    add_glow(base, layer, blur=12)


# ============================================================
# FRAME
# ============================================================

def render_frame(i):

    phase = i / TOTAL_FRAMES

    base = Image.new(
        "RGBA",
        (WIDTH, HEIGHT),
        BG,
    )

    draw_background(base, phase)

    draw_light_cylinder(base)

    draw_field_lines(base, phase)

    draw_particles(base, phase)

    draw_beams(base, phase)

    draw_polar_caps(base, phase)

    draw_star(base, phase)

    return np.array(base.convert("RGB"))


# ============================================================
# RENDER
# ============================================================

frames = []

print("[START] Pulsar Magnetosphere")

for i in range(TOTAL_FRAMES):

    if i % FPS == 0:
        print(
            f"[RENDER] frame {i}/{TOTAL_FRAMES}"
        )

    frames.append(render_frame(i))

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
)

print()
print(f"[SAVED] {out_file.resolve()}")

[START] Pulsar Magnetosphere
[RENDER] frame 0/192


/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_4539/3863527501.py:404: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr, "RGBA")
/var/folders/_b/cfj7mly10r9f3nkywrlqnl300000gn/T/ipykernel_4539/3863527501.py:542: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(


[RENDER] frame 24/192
[RENDER] frame 48/192
[RENDER] frame 72/192
[RENDER] frame 96/192
[RENDER] frame 120/192
[RENDER] frame 144/192
[RENDER] frame 168/192


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[SAVED] /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/pulsar_magnetosphere/pulsar_magnetosphere.webm


[out#0/webm @ 0x15ae25e50] video:2037KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.493577%
frame=  192 fps= 44 q=32.0 Lsize=    2047KiB time=00:00:08.00 bitrate=2095.7kbits/s speed=1.81x    
